# Facility Location on the Adaptive Synthetic Graph

This notebook loads the synthetic dataset built from the `test4` logic, computes demand-to-hub travel costs on the adaptive coarse graph, and solves a Gurobi facility-location model.

The notebook solves the deterministic core model for `y`, `u`, and `x`. The scenario variable `w_s` from the robust formulation is left for a later extension once scenario data are defined.

In [ ]:
from pathlib import Path

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from synthetic_data import build_synthetic_dataset, load_saved_synthetic_bundle


In [ ]:
bundle_path = Path("data/synthetic/ukraine_synthetic_bundle.pkl")

if bundle_path.exists():
    bundle = load_saved_synthetic_bundle(bundle_path)
    print(f"Loaded saved synthetic bundle from {bundle_path}")
else:
    print("Saved synthetic bundle not found; building it now...")
    bundle = build_synthetic_dataset(generate_visuals=True)

filtered_graph = bundle["graphs"]["filtered_graph"]
CG = bundle["graphs"]["adaptive_graph"]
demand_nodes = bundle["demand_nodes"].copy()
summary = bundle["summary"]
cost_details = bundle["metadata"]["cost_details"]

print(f"Filtered road graph: {filtered_graph.number_of_nodes()} nodes, {filtered_graph.number_of_edges()} edges")
print(f"Adaptive CG: {CG.number_of_nodes()} nodes, {CG.number_of_edges()} edges")
print(f"Demand-node rows: {len(demand_nodes)}")


## Modeling notes

A couple of practical additions are needed to make the MIP non-trivial:

- We add a **demand satisfaction** constraint `sum_j x_ij = d_i` for each demand node `i`, otherwise the model could choose `x = 0` everywhere.
- We interpret the service-threshold constraint as a **weighted average travel-time cap**:
  `sum_(i,j) c_ij x_ij <= T * total_demand`.
  This is the natural scaling when `x_ij` is flow/assignment amount.
- We use `n_events` from the clustered ACLED demand nodes as the demand magnitude `d_i` by default. That is easy to swap later.

In [ ]:
# Aggregate demand in case multiple snapped demand rows land on the same coarse node.
demand_df = (
    demand_nodes.groupby("coarse_node", as_index=False)
    .agg(
        demand_amount=("n_events", "sum"),
        total_fatalities=("total_fatalities", "sum"),
        plot_lat=("plot_lat", "first"),
        plot_lon=("plot_lon", "first"),
    )
    .sort_values("coarse_node")
    .reset_index(drop=True)
)

D = demand_df["coarse_node"].tolist()     # demand nodes on the adaptive graph
N = list(CG.nodes())                        # candidate facility nodes
demand = dict(zip(demand_df["coarse_node"], demand_df["demand_amount"]))
a = {j: float(CG.nodes[j]["a_i"]) for j in N}
b = {j: float(CG.nodes[j]["b_i"]) for j in N}

print("CG stats")
print("-" * 60)
print(f"Nodes: {len(N)}")
print(f"Edges: {CG.number_of_edges()}")
print(f"Demand nodes used in optimization: {len(D)}")
print(f"Total demand (sum of n_events): {sum(demand.values()):.2f}")
print(f"a_i range: {min(a.values()):.2f} to {max(a.values()):.2f}")
print(f"b_i range: {min(b.values()):.2f} to {max(b.values()):.2f}")
print("Zone counts:", pd.Series([CG.nodes[n]["zone"] for n in N]).value_counts().to_dict())

demand_df.head()


In [ ]:
# Compute c_ij as shortest-path travel time on the adaptive coarse graph.
# Travel times on edges are in seconds, so we convert to hours for readability.

c = {}
for i in D:
    lengths = nx.single_source_dijkstra_path_length(CG, i, weight="travel_time")
    for j in N:
        if j not in lengths:
            raise ValueError(f"Demand node {i} cannot reach candidate node {j} on the adaptive graph")
        c[(i, j)] = float(lengths[j] / 3600.0)

cost_matrix = pd.DataFrame(
    [[c[(i, j)] for j in N] for i in D],
    index=D,
    columns=N,
)

print("Travel-time matrix stats (hours)")
print("-" * 60)
print(f"min = {cost_matrix.min().min():.3f}")
print(f"mean = {cost_matrix.values.mean():.3f}")
print(f"max = {cost_matrix.max().max():.3f}")

cost_matrix.iloc[: min(5, len(D)), : min(8, len(N))]


In [ ]:
# Core deterministic facility-location MIP
# ----------------------------------------
# min sum_j (a_j y_j + b_j u_j)
# s.t.
#   u_j <= M y_j
#   sum_i x_ij <= u_j
#   sum_j x_ij = d_i                  for each demand i
#   sum_(i,j) c_ij x_ij <= T * total_demand
#
# T is just an initial guess and is easy to change later.

total_demand = float(sum(demand.values()))
M = total_demand
T = 8.0   # guessed average travel-time threshold in hours; tune later

model = gp.Model("adaptive_facility_location")

y = model.addVars(N, vtype=GRB.BINARY, name="y")
u = model.addVars(N, lb=0.0, vtype=GRB.CONTINUOUS, name="u")
x = model.addVars(D, N, lb=0.0, vtype=GRB.CONTINUOUS, name="x")

model.setObjective(
    gp.quicksum(a[j] * y[j] + b[j] * u[j] for j in N),
    GRB.MINIMIZE,
)

# [1] Capacity can only be provisioned at opened facilities.
model.addConstrs((u[j] <= M * y[j] for j in N), name="open_capacity_link")

# [2] Total assigned demand into facility j cannot exceed provisioned capacity u_j.
model.addConstrs((gp.quicksum(x[i, j] for i in D) <= u[j] for j in N), name="facility_capacity")

# Extra demand-balance constraint so every demand node is fully served.
model.addConstrs((gp.quicksum(x[i, j] for j in N) == demand[i] for i in D), name="demand_balance")

# [3] Weighted-average travel time must stay below T.
model.addConstr(
    gp.quicksum(c[(i, j)] * x[i, j] for i in D for j in N) <= T * total_demand,
    name="service_threshold",
)

model.Params.OutputFlag = 1
model.optimize()

print("\nModel status:", model.Status)
print(f"Objective value: {model.ObjVal:.3f}" if model.SolCount > 0 else "No feasible solution found")
print(f"T = {T:.2f} hours, M = {M:.2f}")


In [ ]:
if model.SolCount == 0:
    raise RuntimeError("Gurobi did not return a feasible solution. Try relaxing T.")

open_hubs = [j for j in N if y[j].X > 0.5]
hub_capacity = {j: u[j].X for j in open_hubs}
assignment_rows = []
for i in D:
    for j in N:
        flow = x[i, j].X
        if flow > 1e-6:
            assignment_rows.append(
                {
                    "demand_node": i,
                    "hub_node": j,
                    "flow": flow,
                    "travel_time_hr": c[(i, j)],
                }
            )

assignments = pd.DataFrame(assignment_rows).sort_values(["demand_node", "flow"], ascending=[True, False])
primary_assignments = assignments.drop_duplicates(subset=["demand_node"]).copy()

avg_travel_time = sum(c[(i, j)] * x[i, j].X for i in D for j in N) / total_demand
used_capacity = {j: sum(x[i, j].X for i in D) for j in open_hubs}

print("Solution summary")
print("-" * 60)
print(f"Opened hubs: {len(open_hubs)} / {len(N)}")
print(f"Average weighted travel time: {avg_travel_time:.3f} hr")
print(f"Total fixed cost: {sum(a[j] * y[j].X for j in N):.3f}")
print(f"Total capacity cost: {sum(b[j] * u[j].X for j in N):.3f}")
print(f"Total objective: {model.ObjVal:.3f}")

print("\nOpened hubs")
for j in sorted(open_hubs, key=lambda node: hub_capacity[node], reverse=True):
    attrs = CG.nodes[j]
    print(
        f"  Hub {j}: capacity={hub_capacity[j]:.2f}, used={used_capacity[j]:.2f}, "
        f"a_i={attrs['a_i']:.2f}, b_i={attrs['b_i']:.2f}, zone={attrs['zone']}, members={attrs['member_count']}"
    )

print("\nTop primary assignments")
display(primary_assignments.head(10))


In [ ]:
# Visualize the facility-location solution on top of the full filtered road graph.

road_pos = {n: (filtered_graph.nodes[n]["x"], filtered_graph.nodes[n]["y"]) for n in filtered_graph.nodes()}
fig, ax = plt.subplots(figsize=(14, 9))

nx.draw_networkx_edges(filtered_graph, road_pos, ax=ax, edge_color="gray", alpha=0.08, width=0.35)

hub_x = [CG.nodes[j]["lon"] for j in open_hubs]
hub_y = [CG.nodes[j]["lat"] for j in open_hubs]
hub_s = [140 + 34 * np.sqrt(max(hub_capacity[j], 0.0)) for j in open_hubs]
ax.scatter(
    hub_x,
    hub_y,
    s=hub_s,
    c="forestgreen",
    edgecolors="black",
    linewidths=0.9,
    zorder=5,
    label="Open facility",
)

ax.scatter(
    demand_df["plot_lon"],
    demand_df["plot_lat"],
    marker="*",
    c="royalblue",
    s=160,
    zorder=6,
    label="Demand node",
)

ax.scatter(
    [CG.nodes[n]["lon"] for n in CG.nodes()],
    [CG.nodes[n]["lat"] for n in CG.nodes()],
    s=10,
    c="#c8b27a",
    alpha=0.35,
    zorder=4,
    label="Adaptive CG node",
)

ax.set_title("Facility-location solution over the full filtered road graph")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()
